# 05 - Module 2: Canny fine-tuned diffusion (no synthetic damage)
Trains a rank-8 LoRA on the **SD 1.5 inpainting UNet** with a frozen **Canny ControlNet** (`lllyasviel/control_v11p_sd15_canny`), using **all 5,096 clean MuralDH murals and their Canny edge maps only**. No synthetic damage is added: for every mural the model is asked to paint the **whole** mural from its edge map (mask = whole image), so it learns how Dunhuang murals are painted along their outlines.

In the restoration pipeline this LoRA is used in Stage 2, where the Canny map of the Stage-1 result guides the re-painting of the damaged region.

**Settings:** Accelerator = **GPU T4**, Internet = **ON**. Inputs = `MuralDH` (required) and `MuralDH-Canny` (optional; if missing, the same Canny maps (100/200) are computed on the fly). About 45-60 min.

Output: `/kaggle/working/lora_canny/dunhuang_canny_lora.safetensors`.


In [ ]:
# Kaggle ships torchao 0.10 which peft rejects; we don't use torchao, so switch that dispatcher off.
try:
    import peft.tuners.lora.torchao as _pt
    _pt.is_torchao_available = lambda: False
except Exception as e:
    print("torchao patch skipped:", e)

import os, re, json, math, random, time, gc, glob
import numpy as np, cv2, torch, torch.nn.functional as F
from pathlib import Path
from PIL import Image
import diffusers, peft, transformers
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| peft", peft.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4)"

IMAGE_DIR = Path("/kaggle/input/datasets/balamsd07/muraldh/Mural512/Mural512")
if not IMAGE_DIR.exists():                                     # fall back to searching the inputs
    IMAGE_DIR = Path(next(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/img_0crop_0_0.png", recursive=True)))
OUT = Path("/kaggle/working/lora_canny"); OUT.mkdir(parents=True, exist_ok=True)
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"
CN_ID = "lllyasviel/control_v11p_sd15_canny"
CAPTION = "dunhuang_mural_style, a Dunhuang Buddhist mural painting, mineral pigments, aged plaster wall"
CANNY_LOW, CANNY_HIGH = 100, 200                               # same as notebook 03 and the project
SEED, STEPS, BATCH, ACCUM, LR, RANK = 42, 1000, 2, 2, 1e-4, 8
SAVE_EVERY = 250
dev = torch.device("cuda")
print("images:", IMAGE_DIR)


In [ ]:
# ---- all MuralDH crops are used for training (no held-out split, no synthetic damage) ----
pat = re.compile(r"img_(\d+)crop_(\d+)_(\d+)\.png$")
files = sorted(p for p in IMAGE_DIR.glob("*.png") if pat.match(p.name))
train_files = files
print("training murals:", len(train_files))

# ---- Canny maps: use the MuralDH-Canny dataset if attached, else compute (identical settings) ----
CANNY_DIR = None
for p in glob.glob("/kaggle/input/**/canny/img_0crop_0_0.png", recursive=True):
    CANNY_DIR = Path(os.path.dirname(p)); break
print("Canny source:", CANNY_DIR or "computed on the fly (MuralDH-Canny not attached)")

def get_canny(path):
    if CANNY_DIR is not None and (CANNY_DIR / path.name).exists():
        return cv2.imread(str(CANNY_DIR / path.name), 0)
    gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, CANNY_LOW, CANNY_HIGH)
print("edge pixels in one map: %.1f%%" % ((get_canny(train_files[0]) > 0).mean() * 100))


In [ ]:
class MuralCannyDS(torch.utils.data.Dataset):
    """clean mural + its Canny map. No damage: the 'mask' is the whole image."""
    def __init__(self, files): self.files = files
    def __len__(self): return 10**6
    def __getitem__(self, i):
        rng = np.random.default_rng((i * 2654435761 + int(time.time()*1000)) % (2**32))
        p = self.files[int(rng.integers(len(self.files)))]
        img = np.asarray(Image.open(p).convert("RGB").resize((512, 512)))
        edge = cv2.resize(get_canny(p), (512, 512), interpolation=cv2.INTER_NEAREST)
        if rng.random() < 0.5: img, edge = img[:, ::-1], edge[:, ::-1]
        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 127.5 - 1
        cond = torch.from_numpy(np.ascontiguousarray(edge > 0).astype(np.float32))[None].repeat(3, 1, 1)   # ControlNet wants 3ch in [0,1]
        return dict(pixel=x, cond=cond, drop=bool(rng.random() < 0.1))

dl = torch.utils.data.DataLoader(MuralCannyDS(train_files), batch_size=BATCH, num_workers=2, pin_memory=True)
b = next(iter(dl)); print(b["pixel"].shape, b["cond"].shape)


In [ ]:
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler, ControlNetModel, StableDiffusionControlNetInpaintPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig
from diffusers.training_utils import cast_training_params

tok = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_enc = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to(dev).eval()
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float16).to(dev).eval()
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=torch.float16).to(dev)
controlnet = ControlNetModel.from_pretrained(CN_ID, torch_dtype=torch.float16).to(dev).eval()
sched = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
assert unet.config.in_channels == 9

def enc(text):
    ids = tok(text, padding="max_length", max_length=tok.model_max_length, truncation=True, return_tensors="pt").input_ids.to(dev)
    with torch.no_grad(): return text_enc(ids)[0]
emb_cap, emb_null = enc(CAPTION), enc("")
del text_enc; gc.collect(); torch.cuda.empty_cache()

for m in (vae, unet, controlnet): m.requires_grad_(False)
unet.add_adapter(LoraConfig(r=RANK, lora_alpha=RANK, init_lora_weights="gaussian", target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
cast_training_params(unet, dtype=torch.float32)          # LoRA weights fp32, frozen base fp16
unet.enable_gradient_checkpointing()
params = [p for p in unet.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in params))
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler()


In [ ]:
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

def save_lora(path):
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    sd = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionControlNetInpaintPipeline.save_lora_weights(str(path), unet_lora_layers=sd)   # keys get the 'unet.' prefix

unet.train(); step, micro, hist, t0 = 0, 0, [], time.time()
it = iter(dl); run_loss = 0.0
while step < STEPS:
    b = next(it)
    x = b["pixel"].to(dev, torch.float16)
    cond = b["cond"].to(dev, torch.float16)
    with torch.no_grad():
        lat = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        mlat = vae.encode(torch.zeros_like(x)).latent_dist.sample() * vae.config.scaling_factor   # nothing kept: paint the whole mural
    mask_l = torch.ones_like(lat[:, :1])                                                          # mask = whole image
    noise = torch.randn_like(lat); t = torch.randint(0, sched.config.num_train_timesteps, (lat.shape[0],), device=dev).long()
    noisy = sched.add_noise(lat, noise, t)
    emb = torch.cat([emb_null if d else emb_cap for d in b["drop"]], 0).to(torch.float16)
    with torch.autocast("cuda", dtype=torch.float16):
        with torch.no_grad():                                   # frozen ControlNet: edge features only
            down_res, mid_res = controlnet(noisy, t, encoder_hidden_states=emb, controlnet_cond=cond,
                                           conditioning_scale=1.0, return_dict=False)
        pred = unet(torch.cat([noisy, mask_l, mlat], 1), t, encoder_hidden_states=emb,
                    down_block_additional_residuals=[r for r in down_res],
                    mid_block_additional_residual=mid_res).sample
    mse = (pred.float() - noise.float()) ** 2
    loss = mse.mean() / ACCUM
    scaler.scale(loss).backward(); run_loss += loss.item(); micro += 1
    if micro % ACCUM == 0:
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        hist.append(run_loss); run_loss = 0.0
        if step % 25 == 0: print(f"step {step}/{STEPS}  loss(avg25) {np.mean(hist[-25:]):.4f}  {(time.time()-t0)/60:.1f} min", flush=True)
        if step % SAVE_EVERY == 0: save_lora(OUT / f"ckpt-{step}"); json.dump(hist, open(OUT/"loss_history.json", "w"))
save_lora(OUT / "final"); json.dump(hist, open(OUT/"loss_history.json", "w"))
import shutil; shutil.copy(OUT/"final"/"pytorch_lora_weights.safetensors", OUT/"dunhuang_canny_lora.safetensors")
print("done in", round((time.time()-t0)/60, 1), "min")


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3)); plt.plot(np.convolve(hist, np.ones(25)/25, mode="valid")); plt.title("Canny LoRA training loss (25-step average)")
plt.savefig(OUT/"loss.png", dpi=120); plt.show()
from safetensors import safe_open
with safe_open(str(OUT/"dunhuang_canny_lora.safetensors"), "pt") as f: ks = list(f.keys())
print(len(ks), "tensors; all start with 'unet.':", all(k.startswith("unet.") for k in ks))


In [ ]:
# ---- quick visual check: paint murals from their Canny maps, ControlNet without vs with the Canny LoRA ----
del unet, opt; gc.collect(); torch.cuda.empty_cache()
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(MODEL_ID, controlnet=controlnet, torch_dtype=torch.float16,
                                                                safety_checker=None, requires_safety_checker=False).to(dev)
pipe.set_progress_bar_config(disable=True)
sel = random.Random(1).sample(train_files, 4); full = Image.fromarray(np.full((512, 512), 255, np.uint8)); rows = {}
for name in ["base", "canny_lora"]:
    if name == "canny_lora": pipe.load_lora_weights(str(OUT/"final"), weight_name="pytorch_lora_weights.safetensors"); pipe.fuse_lora(lora_scale=1.0)
    rows[name] = []
    for p in sel:
        e = cv2.resize(get_canny(p), (512, 512), interpolation=cv2.INTER_NEAREST)
        out = pipe(prompt=CAPTION, image=Image.new("RGB", (512, 512)), mask_image=full, control_image=Image.fromarray(np.stack([e]*3, -1)),
                   num_inference_steps=25, guidance_scale=6.0, generator=torch.Generator(dev).manual_seed(7)).images[0]
        rows[name].append(np.asarray(out))
fig, ax = plt.subplots(4, 4, figsize=(14, 14))
for j, p in enumerate(sel):
    ax[0, j].imshow(Image.open(p).convert("RGB").resize((512, 512))); ax[1, j].imshow(get_canny(p), cmap="gray")
    ax[2, j].imshow(rows["base"][j]); ax[3, j].imshow(rows["canny_lora"][j])
for a, t in zip(ax[:, 0], ["original mural", "Canny edges", "ControlNet, no LoRA", "+ Canny LoRA"]): a.set_ylabel(t)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(OUT/"canny_lora_check.png", dpi=110); plt.show()


**Next:** download `lora_canny/dunhuang_canny_lora.safetensors` and `lora_canny/canny_lora_check.png` and copy them into `models/`.